In [1]:
import numpy as np 
import healpy as hp
import pandas as pd
import matplotlib.pyplot as plt
import glob
import seaborn as sns
sns.set_style("darkgrid")
import matplotlib.pyplot as plt
import pandas as pd
# from num2words import num2words as n2
import os
import rasterio
from scipy.interpolate import interp2d
from scipy.integrate import quad
from scipy.stats import multivariate_normal
from src2.country_choice import country_select
from src2.shapecheck import shapecheck as sc
from src2.get_nside import get_nside
from src2.FDrule import FDrule
sns.set(rc={"xtick.bottom": True, "ytick.left": True})
%matplotlib inline
# %matplotlib notebook
import glob

In [2]:
nside = 64

In [3]:
synth_files = glob.glob('./database/*synth.csv')

In [4]:
synth_files

['./database/Argentina_synth.csv',
 './database/Brazil_synth.csv',
 './database/China_synth.csv',
 './database/France_synth.csv',
 './database/Russia_synth.csv',
 './database/Spain_synth.csv']

# Power method

In [5]:
MIDF = pd.read_csv('../../../Spain_LMI.csv')
ODF = pd.read_csv('./database/Spain_synth.csv')


In [6]:
QVals = MIDF['LMQ'].to_numpy()
PVals = MIDF['LMP'].to_numpy()
power = MIDF['EIRP'].to_numpy()
oldlon = MIDF['Longitude'].to_numpy()
oldlat = MIDF['Latitude i'].to_numpy()
latestlon = ODF['Longitude in degrees'].to_numpy()
latestlat = ODF['Latitude in degrees'].to_numpy()

In [7]:
newpix = hp.ang2pix(nside,latestlon,latestlat,lonlat=True)
oldpix = hp.ang2pix(nside,oldlon,oldlat,lonlat=True)


In [8]:
weightage = []
for i in range(len(newpix)):
    idx = newpix[i]
    checkidx = np.where(oldpix==idx)
    if np.sum(checkidx)>0:
        Ps = PVals[checkidx]
        Qs = QVals[checkidx]
        goodidx = Ps<0.05
        if np.sum(goodidx)>0:
            Qfin = Qs[goodidx][0]
            weightage.append(Qfin)
        else:
            weightage.append(0)
    else:
        weightage.append(0)


In [9]:
data2 = power
# Creating histogram to estimate the CDF
hist2, bins2 = np.histogram(data2, bins=FDrule(data2), density=True)
cumulative2 = np.cumsum(hist2 * np.diff(bins2))  # Estimating cumulative distribution function (CDF)

In [10]:
new_power = []
for i in range(len(weightage)):
    if weightage[i]==0:
        new_power.append(np.interp(np.random.uniform(0, 1, 1), (cumulative2), bins2[:-1]))
    elif weightage[i]==1:
        new_power.append(np.interp(np.random.uniform(0.75, 1, 1), (cumulative2), bins2[:-1]))
    elif weightage[i]==2:
        new_power.append(np.interp(np.random.uniform(0.75, 1, 1), (cumulative2), bins2[:-1]))
    elif weightage[i]==3:
        new_power.append(np.interp(np.random.uniform(0, 0.25, 1), (cumulative2), bins2[:-1]))
    elif weightage[i]==4:
        new_power.append(np.interp(np.random.uniform(0, 0.25, 1), (cumulative2), bins2[:-1]))

In [11]:
ODF['EIRP']=new_power